In [1]:
import pandas as pd
import json

In [2]:
file = json.load(open("../output/mimic3/extracted_results.json","r"))

In [68]:
tags = set()
tag = "Education"
cnt = 0
for k,v in file.items():
    for i in v:
        if i['predicted_tags'] == tag:
            try:
                tags.add(i['extracted_predictions']['Education']['extracted_conditions'][0]['education_level'])
            except:
                continue
    

In [88]:

master_ordinal_map = {
    'Adherence':    {'none': 0, 'low': 1, 'normal': 2, 'high': 3, 'None': 999},
    'Education':    {'childhood': 0, 'occupational': 1, 'general': 2, 'high_level': 3, 'None': 999},
    'Employment':   {'retired': 0, 'unemployed': 0, 'on_leave': 1, 'employed': 2, 'None': 999},
    'Financial':    {'poverty': 0, 'constrain': 1, 'normal': 2, 'None': 999},
    'Smoke':        {'current': 0, 'past': 1, 'none': 2, 'None': 999},
    'Social':       {'low': 0, 'medium': 1, 'high': 2, 'None': 999},
    'SubstanceUse': {'current': 0, 'past': 1, 'none': 2, 'None': 999}
}

# A dictionary matching your categories to their respective NLP keys
category_keys = {
    'Adherence': ['adherence_medication', 'adherence_therapy', 'adherence_other'],
    'Education': ["education_level"],
    'Employment': ["employment_status"],
    'Financial': ["financial_status"],
    'Smoke': ["smoke_status"],
    'Social': ['social_family_level', 'social_church_level', 'social_government_level'],
    'SubstanceUse': ['substanceuse_status']
}
employment_to_financial_map = {
    2: 'normal',    # Employed (2) infers Normal finances
    1: 'constrain', # On Leave (1) infers Constrained finances
    0: 'constrain'  # Unemployed/Retired (0) infers Constrained finances
}

In [99]:
import pandas as pd

# 1. THE MASTER ORDINAL MAP
# 0 is always the worst (highest risk). Higher numbers are better.
# 'Unknown' gets 999 so that any real data (0, 1, 2, 3) is smaller and overwrites it.
master_ordinal_map = {
    'Adherence':    {'none': 0, 'low': 1, 'normal': 2, 'high': 3, 'Unknown': 999},
    'Education':    {'childhood': 0, 'occupational': 1, 'general': 2, 'high_level': 3, 'Unknown': 999},
    'Employment':   {'retired': 0, 'unemployed': 0, 'disabled': 0, 'laid off': 0, 'on_leave': 1, 'employed': 2, 'Unknown': 999},
    'Financial':    {'poverty': 0, 'poor': 0, 'constrain': 1, 'normal': 2, 'Unknown': 999},
    'Smoke':        {'current': 0, 'past': 1, 'none': 2, 'Unknown': 999},
    'Social':       {'low': 0, 'medium': 1, 'high': 2, 'Unknown': 999},
    'SubstanceUse': {'current': 0, 'past': 1, 'none': 2, 'Unknown': 999}
}

# 2. NLP KEYS MAP
# Replace these with the actual keys your NLP outputs for each category
category_keys = {
    'Adherence': ['adherence_medication','adherence_other','adherence_therapy'], 
    'Education': ['education_level'],
    'Employment': ['employment_status'],
    'Financial': ['financial_status'],
    'Smoke': ['smoke_status'],
    'Social': ["social_family_level", "social_church_level","social_government_level"], # Supports multiple keys
    'SubstanceUse': ['substanceuse_status']
}


def flatten_sdoh_dataset(raw_data):
    """
    Converts raw SDoH JSON extraction into a wide-format dataframe.
    """
    target_vars = [
        'Adherence', 'Education', 'Employment', 'Financial', 
        'Insurance', 'MentalHealth', 'Smoke', 'Social', 'SubstanceUse'
    ]
    
    processed_records = []

    for patient_id, sentences in raw_data.items():
        # 1. Initialize the patient row
        row = {'Patient_ID': patient_id}
        for var in target_vars:
            row[var] = 0 if var == 'MentalHealth' else 'Unknown'
            
        employed_count = 0
        not_employed_count = 0
        patient_mental_health_types = set()

        # 2. Iterate through each sentence block
        for sentence_obj in sentences:
            predictions = sentence_obj.get('extracted_predictions', {})
            
            for category, details in predictions.items():
                conditions = details.get('extracted_conditions', [])
                
                for condition in conditions:
                    # (Optional) You can check 'Experiencer' here if you want to skip family members
                    # experiencer = condition.get('Experiencer', 'patients')
                    
                    # --- Logic 1: Insurance (Binary Flag) ---
                    if category == 'Insurance':
                        row['Insurance'] = 'Yes'
                        continue
                        
                    # --- Logic 2: Mental Health (Counting unique conditions) ---
                    elif category == 'MentalHealth':
                        # Make sure to capture the actual condition name from the JSON
                        mh_status = condition.get('mentalhealth_type') # Update key if needed
                        if mh_status:
                            patient_mental_health_types.add(mh_status)
                        continue

                    # --- Logic 3: All Ordinal Categories (Smallest Number / Worst Status Wins) ---
                    elif category in master_ordinal_map:
                        keys_to_check = category_keys.get(category, [])
                        specific_map = master_ordinal_map[category]

                        for key in keys_to_check:
                            extracted_status = condition.get(key)
                            
                            if extracted_status:
                                
                                # Special counters for Employment to deduce Financial status later
                                if category == 'Employment':
                                    if extracted_status == 'employed':
                                        employed_count += 1
                                    elif extracted_status in ['unemployed', 'retired', 'laid off', 'disabled']:
                                        not_employed_count += 1

                                # The Universal "Smallest Wins" Update
                                current_status = row[category]
                                weight_extracted = specific_map.get(extracted_status, 999)
                                weight_current = specific_map.get(current_status, 999)
                                
                                # If the new status is a lower number (higher risk), it overwrites the old one
                                if weight_extracted < weight_current:
                                    row[category] = extracted_status


        # 3. Finalize row after reading all sentences
        row['MentalHealth'] = len(patient_mental_health_types)

        # Deduce Financial status from the family employment counters
        deduced_financial = None
        if employed_count > 0:
            deduced_financial = 'normal'
        elif not_employed_count >= 2:
            deduced_financial = 'poor'
        elif not_employed_count == 1:
            deduced_financial = 'constrain'
            
        # Apply deduced financial status (Only if it is WORSE than what NLP directly extracted)
        if deduced_financial:
            current_financial = row['Financial']
            weight_deduced = master_ordinal_map['Financial'].get(deduced_financial, 999)
            weight_current = master_ordinal_map['Financial'].get(current_financial, 999)
            
            if weight_deduced < weight_current:
                row['Financial'] = deduced_financial

        processed_records.append(row)

    # 4. Convert to Pandas DataFrame
    df = pd.DataFrame(processed_records)
    return df

In [100]:
df_final = flatten_sdoh_dataset(file)


In [112]:
df_final.to_csv("../output/mimic3/sdoh_mimic3.csv")

In [107]:
import pandas as pd
import numpy as np

def encode_sdoh_categories(df):
    """
    Encodes categorical SDoH strings into ordered numeric integers for machine learning.
    Automatically converts 'Unknown' or unmapped strings into np.nan for KNN Imputation.
    """
    # Create a copy to avoid SettingWithCopy warnings
    df_encoded = df.copy()
    
    # 1. Define the ML Encoding Maps (Based on your clinical hierarchy)
    # Note: 'Unknown' is intentionally omitted here so .map() forces it to NaN
    encoding_maps = {
        'Adherence':    {'none': 0, 'low': 1, 'normal': 2, 'high': 3},
        'Education':    {'childhood': 0, 'occupational': 1, 'general': 2, 'high_level': 3},
        'Employment':   {'retired': 0, 'unemployed': 0, 'disabled': 0, 'laid off': 0, 'on_leave': 1, 'employed': 2},
        'Financial':    {'poverty': 0, 'poor': 0, 'constrain': 1, 'normal': 2},
        'Smoke':        {'current': 0, 'past': 1, 'none': 2},
        'Social':       {'low': 0, 'medium': 1, 'high': 2},
        'SubstanceUse': {'current': 0, 'past': 1, 'none': 2}
    }

    # 2. Vectorized Mapping for Ordinal Categories
    for col, mapping in encoding_maps.items():
        if col in df_encoded.columns:
            # Map the values, then explicitly cast to the Nullable Integer type
            df_encoded[col] = df_encoded[col].map(mapping).astype('Int64')

            
    # 3. Encode Binary Categories (Insurance)
    if 'Insurance' in df_encoded.columns:
        df_encoded['Insurance'] = df_encoded['Insurance'].map({'Yes': 1, 'no': 0})
        # If you want missing Insurance to be 0 instead of NaN, uncomment the next line:
        # df_encoded['Insurance'] = df_encoded['Insurance'].fillna(0)

    # 4. Ensure Count Categories are Numeric (MentalHealth)
    if 'MentalHealth' in df_encoded.columns:
        df_encoded['MentalHealth'] = pd.to_numeric(df_encoded['MentalHealth'], errors='coerce')

    return df_encoded

# --- How to use it in your pipeline ---
# df_parsed = flatten_sdoh_dataset(raw_json_data)
# df_ml_ready = encode_sdoh_categories(df_parsed)

In [108]:
df_numerical = encode_sdoh_categories(df_final)
# 

In [110]:
for i in df_numerical.columns[1:]:
    print(df_numerical[i].value_counts())

Adherence
0    58
3    23
1    23
2    16
Name: count, dtype: Int64
Education
3    42
1    23
0    13
2     3
Name: count, dtype: Int64
Employment
0    671
2    356
Name: count, dtype: Int64
Financial
1    620
2    419
0     29
Name: count, dtype: Int64
Insurance
1.0    79
Name: count, dtype: int64
MentalHealth
0    3481
1     103
2      14
4       1
3       1
Name: count, dtype: int64
Smoke
1    902
2    354
0    198
Name: count, dtype: Int64
Social
2    125
0     71
1     56
Name: count, dtype: Int64
SubstanceUse
2    997
1    635
0    452
Name: count, dtype: Int64


In [111]:
df_numerical.to_csv("../output/mimic3/encoded_sdoh_mimic3.csv")

In [36]:
df_numerical.Adherence.value_counts()

Adherence
0    3600
Name: count, dtype: int64